# CSE 151B: Homework 2 Coding
## PyTorch Implementation

Using PyTorch’s `Sequential` model class, build a deep convolutional network to classify handwritten digits in MNIST.

You are only allowed to use the following in your model design:
- Linear Layers
- Conv2D
- MaxPool2D
- BatchNorm2D
- Dropout Layers
- ReLU and Softmax
- Flatten

Your goal is to build a model that achieves **test accuracy ≥ 0.985** with fewer than 1 million parameters.

**Warning**: The modules in your Sequential network should *only* consist of `nn` objects! That means you should not be using `torch.nn.functional` modules or lambda expressions in your Sequential block. Leaving functional/lambda expressions in your model code will result in no credit!

This notebook provides a skeleton layout for you. You may use whatever parts of this notebook you deem necessary; there is no need for you to adhere to the structure. However, during submission, you must carefully follow the zip file formatting as requested; see the bottom of the notebook.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
def get_data_loaders(batch_size=32) -> tuple[DataLoader, DataLoader]:
    '''
    Return the training and testing MNIST dataloaders.
    '''
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    # ========================
    # TODO: create dataloaders
    # ========================
    train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
    return train_loader, test_loader


In [4]:
def build_model(dropout_prob=0.5) -> nn.Module:
    model = nn.Sequential(
        # ==========================
        # TODO: fill in architecture
        # ==========================
        nn.Conv2d(1, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),

        nn.Conv2d(32, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),

        nn.MaxPool2d(kernel_size=2),
        nn.Dropout2d(p=dropout_prob),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),

        nn.Conv2d(64, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),

        nn.MaxPool2d(kernel_size=2),
        nn.Dropout2d(p=dropout_prob),

        nn.Flatten(),

        nn.Linear(64 * 7 * 7, 128),
        nn.ReLU(),
        nn.Dropout(p=dropout_prob),

        nn.Linear(128, 10)
    )
    return model


In [5]:
def check_params():
    model = build_model()
    print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")

check_params()

Number of parameters: 468202


In [6]:
def train(model, optimizer, criterion, train_loader, n_epochs = 1):
    '''
    Train the model for `n_epochs` epochs. Returns none (model is modified in place)
    '''
    model.train()
    # =====================
    # TODO: train the model
    # =====================
    for epoch in range(n_epochs):
        total_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)

            predicted = outputs.argmax(dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        avg_loss = total_loss / total
        train_acc = correct / total

        print(f"Epoch {epoch+1}/{n_epochs} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f}")

In [7]:
def test(model, test_loader):
    '''
    Tests the model. Returns none (you should print the accuracy)
    '''
    model.eval()
    # =================================
    # TODO: evaluate the model accuracy
    # =================================
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            predicted = outputs.argmax(dim=1)

            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    print(f"Test Accuracy: {accuracy:.4f}")
    return accuracy

In [8]:
# try 10 different dropout values
train_loader, test_loader = get_data_loaders()
criterion = nn.CrossEntropyLoss() # TODO: use a criterion/loss function
dropout_values = [i / 10 for i in range(10)]
results = {}

for p in dropout_values:
    model = build_model(dropout_prob=p).to(device)
    optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4) # TODO: use an optimizer
    train(model, optimizer, criterion, train_loader)
    acc =test(model, test_loader)
    results[p] = acc
    torch.save(model, f"hw2_dropout_{p:.1f}.pt")
    torch.save(model, f'hw2_dropout_{p}.pt')

print("Dropout sweep results:")
for p, acc in results.items():
    print(f"p = {p:.1f}: test accuracy = {acc:.4f}")

best_p = max(results, key=results.get)
print(f"Best dropout probability: {best_p}, accuracy: {results[best_p]:.4f}")

  0%|          | 0.00/9.91M [00:00<?, ?B/s]

  1%|          | 65.5k/9.91M [00:00<00:20, 481kB/s]

  2%|▏         | 197k/9.91M [00:00<00:12, 765kB/s] 

  8%|▊         | 819k/9.91M [00:00<00:03, 2.51MB/s]

 17%|█▋        | 1.64M/9.91M [00:00<00:02, 3.90MB/s]

...

Test Accuracy: 0.9688


Epoch 1/1 | Loss: 1.6108 | Train Acc: 0.4052


Test Accuracy: 0.9175


Epoch 1/1 | Loss: 2.3520 | Train Acc: 0.1106


Test Accuracy: 0.1135
Dropout sweep results:
p = 0.0: test accuracy = 0.9778
p = 0.1: test accuracy = 0.9869
p = 0.2: test accuracy = 0.9867
p = 0.3: test accuracy = 0.9858
p = 0.4: test accuracy = 0.9843
p = 0.5: test accuracy = 0.9829
p = 0.6: test accuracy = 0.9761
p = 0.7: test accuracy = 0.9688
p = 0.8: test accuracy = 0.9175
p = 0.9: test accuracy = 0.1135
Best dropout probability: 0.1, accuracy: 0.9869


In [10]:
# find your best model, and train it for 10 epochs
best_p = 0.1 # TODO: fill in your best probability
model = build_model(dropout_prob=best_p).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4) # TODO: use an optimizer
train(model, optimizer, criterion, train_loader, n_epochs = 10)
final_acc = test(model, test_loader)
torch.save(model, "hw2_model.pt")
print(f"Final test accuracy after 10 epochs: {final_acc:.4f}")

Epoch 1/10 | Loss: 0.1159 | Train Acc: 0.9641


Epoch 2/10 | Loss: 0.0521 | Train Acc: 0.9841


Epoch 3/10 | Loss: 0.0418 | Train Acc: 0.9870


Epoch 4/10 | Loss: 0.0362 | Train Acc: 0.9887


Epoch 5/10 | Loss: 0.0326 | Train Acc: 0.9898


...

Epoch 7/10 | Loss: 0.0279 | Train Acc: 0.9915


Epoch 8/10 | Loss: 0.0259 | Train Acc: 0.9920


Epoch 9/10 | Loss: 0.0233 | Train Acc: 0.9924


Epoch 10/10 | Loss: 0.0225 | Train Acc: 0.9931


Test Accuracy: 0.9929
Final test accuracy after 10 epochs: 0.9929


# Submission Instructions

Zip all of your **code** and **model .pt files** into one file, and submit on Gradescope to the respective submission.